# Local/Public Validation Mismatch Audit

This notebook checks why the analog-gated Huber + Hilbert candidate improved the bounded local residual sample but failed to beat the ungated Huber + Hilbert submission publicly. The goal is to decide whether the next work should be deeper hierarchy, particle/state-space models, or stricter validation plus boosted-tree feature learning.


## Audit Design

We compare public leaderboard transfer, same-sample local diagnostic deltas, analog-selection distance behavior, and promotion decisions. The audit intentionally avoids fitting a new predictive model; it is a validation-control step before more complex modeling.


In [1]:
from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

ROOT = Path.cwd()
if not (ROOT / "artifacts").exists():
    ROOT = ROOT.parent.parent.parent

RUN_ID = "2026-06-26__validation_mismatch_audit"
RUN_DIR = ROOT / "artifacts" / "logs" / "runs" / RUN_ID
TABLE_DIR = ROOT / "artifacts" / "tables"
REPORT_DIR = ROOT / "artifacts" / "reports"
for path in [RUN_DIR, TABLE_DIR, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "current_public_baseline": 14.514,
    "analog_public_score": 14.603,
    "analog_submission_ref": 53999839,
    "promotion_gate_rmse": 0.05,
    "strict_cv_gate_rmse": 0.10,
}

with (RUN_DIR / "config.json").open("w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

"Load paths and run configuration."
print(ROOT)
print(RUN_DIR)


C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\logs\runs\2026-06-26__validation_mismatch_audit


## Public Transfer Table


In [2]:
public_rows = [
    {"model": "ridge_alpha10", "family": "linear", "local_rmse": 15.7660, "local_basis": "grouped_cv", "public_lb": 15.905, "ref": 53923555},
    {"model": "random_forest", "family": "tree", "local_rmse": 16.0655, "local_basis": "grouped_cv", "public_lb": 15.873, "ref": 53924036},
    {"model": "fine_tuned_gam_spline", "family": "gam", "local_rmse": 14.8687, "local_basis": "sampled_grouped_cv", "public_lb": 15.055, "ref": 53932883},
    {"model": "huber_glm_slope_core", "family": "robust_glm", "local_rmse": 14.7928, "local_basis": "sampled_grouped_cv", "public_lb": 15.464, "ref": 53952950},
    {"model": "bayesian_linear_student_t", "family": "bayesian_linear", "local_rmse": 14.8449, "local_basis": "sampled_grouped_cv", "public_lb": 15.512, "ref": 53957370},
    {"model": "additive_hsgp_student_t", "family": "bayesian_hsgp", "local_rmse": 14.8235, "local_basis": "sampled_grouped_cv", "public_lb": 15.255, "ref": 53958582},
    {"model": "huber_plus_hilbert_residual", "family": "robust_hilbert", "local_rmse": 14.3586, "local_basis": "sampled_grouped_cv", "public_lb": 14.514, "ref": 53960276},
    {"model": "huber_hilbert_direct_analog_gate", "family": "alignment", "local_rmse": 14.6383, "local_basis": "bounded_residual_sample", "public_lb": 14.603, "ref": 53999839},
]
transfer = pd.DataFrame(public_rows)
transfer["public_rank"] = transfer["public_lb"].rank(method="min")
transfer["local_rank_within_basis"] = transfer.groupby("local_basis")["local_rmse"].rank(method="min")
transfer["public_minus_local"] = transfer["public_lb"] - transfer["local_rmse"]
transfer["public_delta_vs_huber_hilbert"] = transfer["public_lb"] - CONFIG["current_public_baseline"]
transfer["promote_public"] = transfer["public_lb"] <= CONFIG["current_public_baseline"] - CONFIG["promotion_gate_rmse"]
transfer["interpretation"] = np.select(
    [
        transfer["model"].eq("huber_plus_hilbert_residual"),
        transfer["model"].eq("huber_hilbert_direct_analog_gate"),
        transfer["public_lb"] < 15.0,
    ],
    [
        "current public baseline",
        "local analog gain did not transfer publicly",
        "strong external candidate but not current best",
    ],
    default="historical baseline",
)
transfer = transfer.sort_values(["public_lb", "model"]).reset_index(drop=True)
transfer_path = TABLE_DIR / "validation__local_public_transfer__2026-06-26.csv"
transfer.to_csv(transfer_path, index=False)
shutil.copy2(transfer_path, RUN_DIR / transfer_path.name)
transfer


,model,family,local_rmse,local_basis,public_lb,ref,public_rank,local_rank_within_basis,public_minus_local,public_delta_vs_huber_hilbert,promote_public,interpretation
0,huber_plus_hilbert_residual,robust_hilbert,14.3586,sampled_grouped_cv,14.514,53960276,1.0,1.0,0.1554,0.000,False,current public baseline
1,huber_hilbert_direct_analog_gate,alignment,14.6383,bounded_residual_sample,14.603,53999839,2.0,1.0,-0.0353,0.089,False,local analog gain did not transfer publicly
2,fine_tuned_gam_spline,gam,14.8687,sampled_grouped_cv,15.055,53932883,3.0,5.0,0.1863,0.541,False,historical baseline
3,additive_hsgp_student_t,bayesian_hsgp,14.8235,sampled_grouped_cv,15.255,53958582,4.0,3.0,0.4315,0.741,False,historical baseline
4,huber_glm_slope_core,robust_glm,14.7928,sampled_grouped_cv,15.464,53952950,5.0,2.0,0.6712,0.950,False,historical baseline
5,bayesian_linear_student_t,bayesian_linear,14.8449,sampled_grouped_cv,15.512,53957370,6.0,4.0,0.6671,0.998,False,historical baseline
6,random_forest,tree,16.0655,grouped_cv,15.873,53924036,7.0,2.0,-0.1925,1.359,False,historical baseline
7,ridge_alpha10,linear,15.7660,grouped_cv,15.905,53923555,8.0,1.0,0.1390,1.391,False,historical baseline


## Analog Local Gain vs Public Loss


In [3]:
summary = pd.read_csv(TABLE_DIR / "model__alignment_hsgp_residual_summary__2026-06-23.csv")
zone = pd.read_csv(TABLE_DIR / "model__alignment_hsgp_residual_zone_summary__2026-06-23.csv")
well_diag = pd.read_csv(TABLE_DIR / "model__alignment_hsgp_residual_well_diagnostics__2026-06-23.csv")
selection = pd.read_csv(TABLE_DIR / "model__analog_selection_refinement_selection_diagnostics__2026-06-23.csv")

best_candidate = "log_plus_trajectory_k1_progress_gate"
best_selection = selection.loc[selection["candidate"].eq(best_candidate)].copy()
best_selection["selected_analog_count"] = best_selection["selected_analogs"].fillna("").astype(str).str.split(";").map(lambda x: len([v for v in x if v]))
analog_well = best_selection.merge(well_diag, on=["fold", "wellname"], how="left")
analog_well["bias_improved"] = analog_well["abs_bias_delta"] < 0
analog_well["distance_bin"] = pd.qcut(analog_well["mean_distance"], q=5, labels=["closest", "near", "middle", "far", "farthest"], duplicates="drop")

distance_bins = (
    analog_well.groupby("distance_bin", observed=True)
    .agg(
        n_wells=("wellname", "nunique"),
        mean_distance=("mean_distance", "mean"),
        median_distance=("mean_distance", "median"),
        mean_alignment_correction=("mean_alignment_correction", "mean"),
        mean_abs_bias_delta=("abs_bias_delta", "mean"),
        median_abs_bias_delta=("abs_bias_delta", "median"),
        share_wells_bias_improved=("bias_improved", "mean"),
    )
    .reset_index()
)

distance_path = TABLE_DIR / "validation__analog_distance_bins__2026-06-26.csv"
distance_bins.to_csv(distance_path, index=False)
shutil.copy2(distance_path, RUN_DIR / distance_path.name)

worst_wells = analog_well.sort_values(["abs_bias_delta", "mean_distance"], ascending=[False, False]).head(25)
worst_path = TABLE_DIR / "validation__analog_worst_well_sample__2026-06-26.csv"
worst_wells.to_csv(worst_path, index=False)
shutil.copy2(worst_path, RUN_DIR / worst_path.name)

print(summary.head(8).to_string(index=False))
distance_bins


                     candidate               kind  rmse_mean  rmse_std  mae_mean  mean_correction  std_correction  max_abs_correction  n_rows  n_wells  rmse_delta_vs_sample_baseline
            direct_analog_gate      direct_analog  14.638289  0.411922 10.353734        -0.043459        1.207596            8.494656    5400      773                      -0.100256
    analog_row_ridge_a300_s050   analog_row_ridge  14.662878  0.437523 10.387546         0.118941        1.085605           14.000000    5400      773                      -0.075667
   analog_row_ridge_a1000_s050   analog_row_ridge  14.667192  0.439496 10.385563         0.120036        1.009437           14.000000    5400      773                      -0.071353
 analog_poly2_ridge_a1500_s035 analog_poly2_ridge  14.668348  0.439053 10.376764         0.080443        0.948466           14.000000    5400      773                      -0.070197
    analog_row_huber_a010_s050   analog_row_huber  14.684683  0.436384 10.404055         0

,distance_bin,n_wells,mean_distance,median_distance,mean_alignment_correction,mean_abs_bias_delta,median_abs_bias_delta,share_wells_bias_improved
0,closest,155,0.247443,0.254888,-0.078706,-0.070585,-0.084725,0.541935
1,near,154,0.333887,0.334908,-0.031357,0.027845,-0.007592,0.500000
2,middle,155,0.396311,0.397252,0.071599,-0.217731,-0.145761,0.606452
3,far,154,0.476318,0.474547,-0.118637,-0.110667,-0.087814,0.564935
4,farthest,155,0.674709,0.618877,-0.059641,-0.084340,-0.067676,0.561290


## Promotion Decisions


In [4]:
decisions = pd.DataFrame([
    {"track": "hierarchy", "evidence": "empirical Bayes offsets", "local_delta_rmse": -0.0008, "public_delta_rmse": np.nan, "decision": "pause", "reason": "gain is far below 0.05 RMSE gate"},
    {"track": "hierarchy", "evidence": "residual shape screen", "local_delta_rmse": -0.0003, "public_delta_rmse": np.nan, "decision": "pause", "reason": "gain is far below 0.05 RMSE gate"},
    {"track": "hierarchy", "evidence": "contextual Huber interactions", "local_delta_rmse": -0.0064, "public_delta_rmse": np.nan, "decision": "pause", "reason": "weak local positive signal but not enough for deeper hierarchy"},
    {"track": "alignment", "evidence": "direct analog gate", "local_delta_rmse": -0.1003, "public_delta_rmse": 14.603 - 14.514, "decision": "do_not_promote", "reason": "local gain did not transfer; public score worsened by 0.089"},
    {"track": "particle_state_space", "evidence": "latent trajectory idea", "local_delta_rmse": np.nan, "public_delta_rmse": np.nan, "decision": "defer", "reason": "model class is plausible but validation mismatch is the current bottleneck"},
    {"track": "validation", "evidence": "local/public rank mismatch", "local_delta_rmse": np.nan, "public_delta_rmse": np.nan, "decision": "promote_next", "reason": "stricter CV is needed before boosted trees or deeper Bayesian models"},
    {"track": "boosted_trees", "evidence": "rich feature learner after strict CV", "local_delta_rmse": np.nan, "public_delta_rmse": np.nan, "decision": "next_model_family", "reason": "try flexible interactions only after validation controls are fixed"},
])
decision_path = TABLE_DIR / "validation__promotion_decisions__2026-06-26.csv"
decisions.to_csv(decision_path, index=False)
shutil.copy2(decision_path, RUN_DIR / decision_path.name)
decisions


,track,evidence,local_delta_rmse,public_delta_rmse,decision,reason
0,hierarchy,empirical Bayes offsets,-0.0008,NaN,pause,gain is far below 0.05 RMSE gate
1,hierarchy,residual shape screen,-0.0003,NaN,pause,gain is far below 0.05 RMSE gate
2,hierarchy,contextual Huber interactions,-0.0064,NaN,pause,weak local positive signal but not enough for ...
3,alignment,direct analog gate,-0.1003,0.089,do_not_promote,local gain did not transfer; public score wors...
4,particle_state_space,latent trajectory idea,NaN,NaN,defer,model class is plausible but validation mismat...
5,validation,local/public rank mismatch,NaN,NaN,promote_next,stricter CV is needed before boosted trees or ...
6,boosted_trees,rich feature learner after strict CV,NaN,NaN,next_model_family,try flexible interactions only after validatio...


## Report Export


In [5]:
best_public = transfer.iloc[0]
analog_row = transfer.loc[transfer["model"].eq("huber_hilbert_direct_analog_gate")].iloc[0]
analog_public_delta = analog_row["public_delta_vs_huber_hilbert"]
local_analog_gain = -float(summary.loc[summary["candidate"].eq("direct_analog_gate"), "rmse_delta_vs_sample_baseline"].iloc[0])
improved_share = float(distance_bins["share_wells_bias_improved"].mean())

findings = pd.DataFrame([
    {"finding": "Current public baseline remains Huber + additive Hilbert residual", "value": f"{best_public['public_lb']:.3f}", "implication": "Keep 14.514 as the external score to beat."},
    {"finding": "Analog gate worsened public LB despite local diagnostic gain", "value": f"public delta {analog_public_delta:.3f}; local gain {local_analog_gain:.4f}", "implication": "Do not promote analog gate; investigate validation mismatch."},
    {"finding": "Analog well bias improvement is mixed", "value": f"mean distance-bin improvement share {improved_share:.3f}", "implication": "Analog quality needs stricter holdout tests before reuse."},
    {"finding": "Hierarchy gains are below gate", "value": "0.0003 to 0.0064 RMSE in shallow screens", "implication": "Pause hierarchy until stronger descriptors or strict folds justify it."},
    {"finding": "Particle/state-space models are deferred", "value": "no validation evidence yet", "implication": "Consider later as latent trajectory modeling, not before CV repair."},
])
findings_path = TABLE_DIR / "validation__mismatch_findings__2026-06-26.csv"
findings.to_csv(findings_path, index=False)
shutil.copy2(findings_path, RUN_DIR / findings_path.name)

report = f"""# Validation Mismatch Audit - 2026-06-26

## Summary

The Huber + additive Hilbert residual submission remains the public baseline at **14.514**. The direct analog-gated extension scored **14.603** on public LB, worsening the external score by **{analog_public_delta:.3f}** even though the bounded residual sample showed a **{local_analog_gain:.4f} RMSE** local improvement.

## Decision

Pause deeper hierarchy and particle/state-space modeling. The current bottleneck is validation transfer, not model expressiveness. The next notebook should design stricter folds that stress far-analog, clustered/geologic, and high-extrapolation cases before boosted-tree tuning.

## Evidence

- Analog gate submission ref: `53999839`; public LB **14.603**.
- Current public baseline ref: `53960276`; public LB **14.514**.
- Local analog diagnostic was on a bounded residual sample, not the same full local CV used for the original Huber + Hilbert result.
- Shallow hierarchy screens failed the promotion gate with gains below **0.05 RMSE**.

## Next Steps

1. Build `01_stricter_cv_fold_design.ipynb` with cluster, far-analog, and high-extrapolation folds.
2. Re-score Huber + Hilbert and analog-gated variants under those folds.
3. Start LightGBM/XGBoost only after strict folds identify feature families that transfer.
4. Revisit hierarchy or particle models only if strict-fold residuals show coherent well-level or sequential latent structure.
"""
report_path = REPORT_DIR / "validation__mismatch_audit__2026-06-26.md"
report_path.write_text(report, encoding="utf-8")
shutil.copy2(report_path, RUN_DIR / report_path.name)

print(report)


# Validation Mismatch Audit - 2026-06-26

## Summary

The Huber + additive Hilbert residual submission remains the public baseline at **14.514**. The direct analog-gated extension scored **14.603** on public LB, worsening the external score by **0.089** even though the bounded residual sample showed a **0.1003 RMSE** local improvement.

## Decision

Pause deeper hierarchy and particle/state-space modeling. The current bottleneck is validation transfer, not model expressiveness. The next notebook should design stricter folds that stress far-analog, clustered/geologic, and high-extrapolation cases before boosted-tree tuning.

## Evidence

- Analog gate submission ref: `53999839`; public LB **14.603**.
- Current public baseline ref: `53960276`; public LB **14.514**.
- Local analog diagnostic was on a bounded residual sample, not the same full local CV used for the original Huber + Hilbert result.
- Shallow hierarchy screens failed the promotion gate with gains below **0.05 RMSE**.

## Nex